In [1]:
import os

In [2]:
pwd=os.getcwd()

In [3]:
os.chdir("../")
print(os.getcwd())

/Users/xenan.bilgin/Projects/MLops/mlops-project1-wine-quality


In [4]:
import pandas as pd
data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [6]:
# Create schema.yaml file for data validation
data.isnull().sum()


fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: Path
    all_schema: dict

In [8]:
from src.mlops1_data_science_project import logger
from src.mlops1_data_science_project.utils.common import read_yaml, create_directories
from src.mlops1_data_science_project.constants import *



In [9]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self)-> DataValidationConfig:
        config=self.config.data_validation
        schema = self.schema.COLUMNS
        create_directories([config.root_dir])

        data_validation_config=DataValidationConfig(
            root_dir=config.root_dir,
            unzip_data_dir=config.unzip_data_dir,
            STATUS_FILE=config.STATUS_FILE,
            all_schema=schema
        )
        return data_validation_config

In [11]:
import pandas as pd

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self):
        try:
            validation_status = True

            data = pd.read_csv(self.config.unzip_data_dir)

            all_cols = set(data.columns)
            all_schema_cols = set(self.config.all_schema.keys())

            missing_cols = all_schema_cols - all_cols
            extra_cols = all_cols - all_schema_cols

            with open(self.config.STATUS_FILE, "w") as f:

                if missing_cols:
                    validation_status = False

                    for col in missing_cols:
                        logger.info(f"Missing column in dataset: {col}")
                        f.write(f"Missing column: {col}\n")

                if extra_cols:
                    validation_status = False

                    for col in extra_cols:
                        logger.info(f"Extra column in dataset: {col}")
                        f.write(f"Extra column: {col}\n")

                if validation_status:
                    logger.info("All columns are valid")
                    f.write("Validation status: True\n")
                else:
                    f.write("Validation status: False\n")

            return validation_status

        except Exception as e:
            logger.exception(e)
            raise e

In [13]:
try:
    config=ConfigurationManager()
    data_validation_config=config.get_data_validation_config()
    data_validation=DataValidation(config=data_validation_config)
    validation_status=data_validation.validate_all_columns()
    logger.info(f"Data validation status: {validation_status}")
except Exception as e:
    logger.exception(e)

[2026-05-27 16:49:37,994: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-27 16:49:37,995: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-27 16:49:37,997: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-27 16:49:37,998: INFO: common: created directory at: artifacts]
[2026-05-27 16:49:37,998: INFO: common: created directory at: artifacts/data_validation]
[2026-05-27 16:49:38,000: INFO: 1253244569: All columns are valid]
[2026-05-27 16:49:38,001: INFO: 2729420827: Data validation status: True]
